# Sensor noise-level visualizer

Exercises `sensors/sensors.py` against every `NoiseLevel` preset
(`IDEAL`, `LOW`, `MEDIUM`, `HIGH`) and plots the result so it's easy to
see, side-by-side, how much noise and drift each profile injects.

The notebook walks through:

1. Building the analytical ground truth from `Robot` /
   `TrajectoryGenerator`.
2. Sampling the **IMU accelerometer** at every preset and plotting
   `(ax, ay)` against the ground truth.
3. Plotting the IMU's internal **bias drift** over time.
4. Sampling the **wheel-odometry** sensor at every preset.
5. Showing the **effect on IMU-only dead reckoning** — i.e. why fusion
   matters once you leave the `IDEAL` preset.
6. Demoing the **runtime `set_noise_level` swap** with one IMU instance.
7. Comparing the **configured vs empirical noise std-devs** as a sanity
   check that the presets do what they claim.

In [1]:
import sys
from pathlib import Path

# The notebook lives in `sensors/`; the repo root holds `robot.py` and
# the `sensors` package.  Force the repo root to `sys.path[0]` so
# `from sensors.sensors import ...` resolves to the *package* sibling of
# `robot.py`, even when Jupyter has already prepended the current cell's
# directory (which contains `sensors.py` and would otherwise be picked
# first, binding `sensors` to a *module* rather than a package).
NB_DIR = Path.cwd()
REPO_ROOT = NB_DIR if (NB_DIR / "robot.py").exists() else NB_DIR.parent
repo_root_str = str(REPO_ROOT)
while repo_root_str in sys.path:
    sys.path.remove(repo_root_str)
sys.path.insert(0, repo_root_str)

import numpy as np
import matplotlib.pyplot as plt

from robot import Robot
from sensors.sensors import (
    NoiseLevel,
    ImuSensor,
    WheelOdometrySensor,
    IMU_PRESETS,
    WHEEL_PRESETS,
    make_sensors,
)

LEVELS = list(NoiseLevel)
LEVEL_COLORS = {
    NoiseLevel.IDEAL:  "#2ca02c",
    NoiseLevel.LOW:    "#1f77b4",
    NoiseLevel.MEDIUM: "#ff7f0e",
    NoiseLevel.HIGH:   "#d62728",
}

print("Repo root:", REPO_ROOT)
print("Noise levels:", [lvl.value for lvl in LEVELS])

ModuleNotFoundError: No module named 'sensors.sensors'; 'sensors' is not a package

## 1. Ground truth

Sample the analytical trajectory at the IMU rate so every sensor cell
below has a consistent set of `(position, velocity, acceleration)`
references to compare against.

In [ ]:
DURATION_S = 12.0
IMU_RATE_HZ = 1000.0
DT = 1.0 / IMU_RATE_HZ

times = np.arange(0.0, DURATION_S + DT, DT)

robot = Robot()
states = robot.states_at(times)

truth_accel = np.array([s.acceleration for s in states])
truth_vel   = np.array([s.velocity for s in states])
truth_pos   = np.array([s.position for s in states])

print(f"{len(times)} samples over {DURATION_S:.1f}s at {IMU_RATE_HZ:.0f} Hz")

## 2. IMU accelerometer across noise levels

For each preset we build an `ImuSensor` from `ImuSensor.from_noise_level`,
sample the full trajectory, and store both the noisy reading and the
internal bias trace.  The plot below shows the noisy measurement (in
colour) on top of the analytical truth (in black) for each level.

In [ ]:
def sample_imu(level: NoiseLevel, seed: int = 7) -> tuple[np.ndarray, np.ndarray]:
    """Return (measured_accel, bias_trace) for the whole trajectory."""
    rng = np.random.default_rng(seed)
    imu = ImuSensor.from_noise_level(level, rng, rate_hz=IMU_RATE_HZ)
    accel = np.empty((len(times), 2))
    bias = np.empty((len(times), 2))
    for i, st in enumerate(states):
        accel[i] = imu.measure(st, DT)
        bias[i] = imu.bias
    return accel, bias


imu_traces = {lvl: sample_imu(lvl) for lvl in LEVELS}

fig, axes = plt.subplots(len(LEVELS), 2, figsize=(11, 9), sharex=True)
for row, lvl in enumerate(LEVELS):
    accel, _ = imu_traces[lvl]
    for col, axis_name in enumerate(("ax", "ay")):
        ax = axes[row, col]
        ax.plot(times, truth_accel[:, col], color="k", lw=1.3, label="truth")
        ax.plot(
            times,
            accel[:, col],
            color=LEVEL_COLORS[lvl],
            lw=0.8,
            alpha=0.7,
            label=f"{lvl.value} measurement",
        )
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(axis_name)
        if col == 0:
            ax.set_ylabel(f"{lvl.value}\n[m/s²]")
        ax.legend(loc="upper right", fontsize=8)
axes[-1, 0].set_xlabel("t [s]")
axes[-1, 1].set_xlabel("t [s]")
fig.suptitle("IMU accelerometer: truth vs noisy reading across noise levels", y=1.0)
fig.tight_layout()
plt.show()

## 3. IMU bias drift

`IDEAL` and `LOW` should be effectively flat, `MEDIUM` should wander
visibly, and `HIGH` should look like Brownian motion — the random-walk
variance scales with `bias_random_walk_std²·Δt`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for lvl in LEVELS:
    _, bias = imu_traces[lvl]
    axes[0].plot(times, bias[:, 0], color=LEVEL_COLORS[lvl], lw=1.2, label=lvl.value)
    axes[1].plot(times, bias[:, 1], color=LEVEL_COLORS[lvl], lw=1.2, label=lvl.value)
for ax, name in zip(axes, ("bias_x", "bias_y")):
    ax.set_xlabel("t [s]")
    ax.set_ylabel("[m/s²]")
    ax.set_title(name)
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.suptitle("IMU acceleration-bias drift over time")
fig.tight_layout()
plt.show()

## 4. Wheel odometry across noise levels

Wheel odometry is sampled at a lower rate (here 50 Hz, well below the
IMU's 1 kHz).  Each level adds both a multiplicative `scale_error`
(systematic bias) and additive `velocity_noise_std` white noise, so
`HIGH` consistently over-reads on top of being noisy.

In [ ]:
WHEEL_RATE_HZ = 50.0
WHEEL_DT = 1.0 / WHEEL_RATE_HZ

wheel_times = np.arange(0.0, DURATION_S + WHEEL_DT, WHEEL_DT)
wheel_states = robot.states_at(wheel_times)
wheel_truth_vel = np.array([s.velocity for s in wheel_states])


def sample_wheel(level: NoiseLevel, seed: int = 7) -> np.ndarray:
    rng = np.random.default_rng(seed)
    wheel = WheelOdometrySensor.from_noise_level(level, rng, rate_hz=WHEEL_RATE_HZ)
    return np.array([wheel.measure(s) for s in wheel_states])


wheel_traces = {lvl: sample_wheel(lvl) for lvl in LEVELS}

fig, axes = plt.subplots(len(LEVELS), 2, figsize=(11, 9), sharex=True)
for row, lvl in enumerate(LEVELS):
    meas = wheel_traces[lvl]
    for col, name in enumerate(("vx", "vy")):
        ax = axes[row, col]
        ax.plot(wheel_times, wheel_truth_vel[:, col], color="k", lw=1.3, label="truth")
        ax.plot(
            wheel_times,
            meas[:, col],
            color=LEVEL_COLORS[lvl],
            lw=0.9,
            alpha=0.8,
            label=f"{lvl.value} measurement",
        )
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(name)
        if col == 0:
            ax.set_ylabel(f"{lvl.value}\n[m/s]")
        ax.legend(loc="upper right", fontsize=8)
axes[-1, 0].set_xlabel("t [s]")
axes[-1, 1].set_xlabel("t [s]")
fig.suptitle("Wheel odometry: truth vs noisy reading across noise levels", y=1.0)
fig.tight_layout()
plt.show()

## 5. Effect on IMU-only dead reckoning

Integrating noisy acceleration twice is exactly what an unfiltered
strap-down IMU does, and it's why fusion is necessary.  Even with the
`IDEAL` preset the trajectory matches truth; for `MEDIUM` it wanders;
for `HIGH` it drifts spectacularly within a few seconds.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(truth_pos[:, 0], truth_pos[:, 1], color="k", lw=2.2, label="truth")

for lvl in LEVELS:
    accel, _ = imu_traces[lvl]
    p = truth_pos[0].copy()
    v = truth_vel[0].copy()
    out = np.empty_like(accel)
    for i, a in enumerate(accel):
        p = p + v * DT + 0.5 * a * DT * DT
        v = v + a * DT
        out[i] = p
    ax.plot(
        out[:, 0],
        out[:, 1],
        color=LEVEL_COLORS[lvl],
        lw=1.3,
        alpha=0.9,
        label=lvl.value,
    )

ax.set_xlabel("x [m]")
ax.set_ylabel("y [m]")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title("IMU-only dead reckoning across noise levels")
plt.show()

## 6. Runtime noise-level swap

The sensors expose `set_noise_level(...)` so you can switch profiles
mid-run.  Below, a single `ImuSensor` starts on the `LOW` preset and
flips to `HIGH` at the halfway mark.  The bias is re-sampled at the
swap (a brand-new initial bias is drawn from the new preset's
`bias_initial_std`), then drifts under the much larger random walk.

In [ ]:
rng = np.random.default_rng(11)
imu_swap = ImuSensor.from_noise_level(NoiseLevel.LOW, rng, rate_hz=IMU_RATE_HZ)

swap_idx = len(times) // 2
accel_swap = np.empty((len(times), 2))
bias_swap = np.empty((len(times), 2))
for i, st in enumerate(states):
    if i == swap_idx:
        imu_swap.set_noise_level(NoiseLevel.HIGH)
    accel_swap[i] = imu_swap.measure(st, DT)
    bias_swap[i] = imu_swap.bias

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(times, truth_accel[:, 0], color="k", lw=1.4, label="truth ax")
axes[0].plot(times, accel_swap[:, 0], color="#9467bd", lw=0.8, alpha=0.85,
             label="measured ax")
axes[0].axvline(times[swap_idx], color="gray", ls="--", label="LOW → HIGH swap")
axes[0].set_ylabel("[m/s²]")
axes[0].set_title("IMU acceleration during runtime noise-level swap")
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="upper right")

axes[1].plot(times, bias_swap[:, 0], color="#9467bd", lw=1.4, label="bias_x")
axes[1].plot(times, bias_swap[:, 1], color="#e377c2", lw=1.4, label="bias_y")
axes[1].axvline(times[swap_idx], color="gray", ls="--")
axes[1].set_xlabel("t [s]")
axes[1].set_ylabel("[m/s²]")
axes[1].set_title("Bias drift accelerates after switching to HIGH")
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc="upper right")

fig.tight_layout()
plt.show()

## 7. Configured vs empirical noise

A sanity check: subtract the truth (and, for the IMU, also subtract the
recorded bias trace) to isolate the white-noise component, then
compare its empirical std-dev against the value the preset advertises.
The two should match to within sampling error.

In [ ]:
rows = []
for lvl in LEVELS:
    accel, bias = imu_traces[lvl]
    # Isolate the pure white-noise residual by subtracting both the
    # truth acceleration and the (known) bias trace.
    imu_residual = accel - truth_accel - bias

    meas = wheel_traces[lvl]
    scale = 1.0 + WHEEL_PRESETS[lvl].scale_error
    wheel_residual = meas - scale * wheel_truth_vel

    rows.append(
        {
            "level": lvl.value,
            "imu_cfg":   IMU_PRESETS[lvl].accel_noise_std,
            "imu_emp_x": float(imu_residual[:, 0].std()),
            "imu_emp_y": float(imu_residual[:, 1].std()),
            "wheel_cfg": WHEEL_PRESETS[lvl].velocity_noise_std,
            "wheel_emp_x": float(wheel_residual[:, 0].std()),
            "wheel_emp_y": float(wheel_residual[:, 1].std()),
        }
    )

# Pretty-print, with a graceful fallback if pandas is unavailable.
try:
    import pandas as pd

    df = (
        pd.DataFrame(rows)
        .set_index("level")
        .round(5)
    )
    display(df)
except ImportError:
    header = ("level", "imu_cfg", "imu_emp_x", "imu_emp_y",
              "wheel_cfg", "wheel_emp_x", "wheel_emp_y")
    print("  ".join(f"{h:>11}" for h in header))
    for r in rows:
        print("  ".join(
            f"{r[h]:>11}" if isinstance(r[h], str)
            else f"{r[h]:>11.5f}"
            for h in header
        ))